# Setup (Dependencies & Imports)

Install dependencies, clone the repo, and import required libraries and modules.

In [ ]:
# Install dependencies
!pip install -q transformers torch torchvision numpy pillow requests tqdm


In [ ]:
# Clone the project repo
!git clone https://github.com/gizayceylan/FakeNews.git

# Add it to Python path
import sys
sys.path.append("/content/FakeNews")


In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import torch
import requests
import json

from numpy.linalg import norm
from PIL import Image
from tqdm import tqdm
from google.colab import files

# ML/DL Libraries
from transformers import CLIPProcessor, CLIPModel
from torchvision.datasets.utils import download_url


# Metadata (Dataset Loading & Preparation)

Load the Fakeddit metadata, clean it, and select the subset of samples containing valid image URLs.

In [ ]:
# Load metadata
dataset_url = "https://raw.githubusercontent.com/gizayceylan/FakeNews/main/Datasets/fakeddit_small_2_way.csv"
df = pd.read_csv(dataset_url)
print("Shape original:", df.shape)

# Remove duplicate rows (if any)
df = df.drop_duplicates(keep="first")
print("Shape after removing duplicates:", df.shape)

# Keep entries with valid image URLs
df = df[df["image_url"].notna()].reset_index(drop=True)
print("Shape after removing rows without image_url:", df.shape)

df.head()

In [ ]:
# Check label distribution
label_counts = df['2_way_label'].value_counts().sort_index()
label_percent = df['2_way_label'].value_counts(normalize=True).sort_index() * 100

# Combine into one table for display
balance_df = pd.DataFrame({
    'Label': ['Fake (0)', 'Real (1)'],
    'Count': label_counts.values,
    'Percentage': label_percent.values.round(2)
})

balance_df

In [ ]:
# Make a folder for images
img_dir = "/content/fakeddit_images"
os.makedirs(img_dir, exist_ok=True)


In [ ]:
# Sample N images for testing
N = 300
sample_df = df.sample(N, random_state=42).reset_index(drop=True)


In [ ]:
# Download images from URLs
image_paths = []
failed = 0

for i, url in tqdm(list(enumerate(sample_df["image_url"])), total=len(sample_df)):
    filename = f"{i}.jpg"          # index inside sample
    filepath = os.path.join(img_dir, filename)

    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            with open(filepath, "wb") as f:
                f.write(r.content)
            image_paths.append(filepath)
        else:
            image_paths.append(None)
            failed += 1
    except Exception:
        image_paths.append(None)
        failed += 1

sample_df["image_path"] = image_paths
print("\nFailed downloads:", failed)


In [ ]:
# Keep only successfully downloaded images
image_df = sample_df[sample_df["image_path"].notna()].reset_index(drop=True)
image_df = image_df[['image_path', 'clean_title','2_way_label', 'image_url']].rename(columns={'2_way_label': 'label'})
print("Images available:", len(image_df))
image_df.head()


In [ ]:
# Test a sample
Image.open(image_df["image_path"].iloc[0])


# Visual Concepts (CLIP & ImageNet + Places365)

Extract the concept embeddings with a large vocabulary of ImageNet and Places365.

In [ ]:
# Load CLIP model & processor
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()


In [ ]:
# Encode text using CLIP text encoder
def get_text_embedding(text_list):
    """
    Encodes a list of text labels using the CLIP text encoder.
    Returns a NumPy matrix of shape (N_texts, 512).
    """
    inputs = clip_processor(text=text_list, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        text_emb = clip_model.get_text_features(**inputs)
    return text_emb.cpu().numpy()

In [ ]:
# Download ImageNet-1K class names
url_imagenet = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
download_url(url_imagenet, ".", "imagenet_classes.txt")

with open("imagenet_classes.txt", "r") as f:
    imagenet_labels = [line.strip() for line in f.readlines()]

print("\nImageNet classes loaded:", len(imagenet_labels))
imagenet_labels[:10]


In [ ]:
# Download Places365 category file
url_places = "https://raw.githubusercontent.com/csailvision/places365/master/categories_places365.txt"
download_url(url_places, ".", "places365.txt")

places365_labels = []
with open("places365.txt", "r") as f:
    for line in f:
        raw_label = line.split()[0]         # e.g. "/a/airfield"
        label = raw_label.split("/", 2)[-1] # e.g. "airfield" OR "apartment_building/outdoor"
        label = label.replace("_", " ").replace("/", " ")
        places365_labels.append(label)

print("\nPlaces365 classes loaded:", len(places365_labels))
places365_labels[:10]


In [ ]:
# Combine concept vocabulary
concepts = imagenet_labels + places365_labels
print("Total visual concepts:", len(concepts))


In [ ]:
# Compute CLIP text embeddings
concept_embs = get_text_embedding(concepts)
print("Concept embedding matrix shape:", concept_embs.shape)


# Save Outputs

Save, the image paths, images and the visual concepts for reproducibility.

In [ ]:
# Create directory for output files
output_dir = "/content/master_pipeline_assets"
os.makedirs(output_dir, exist_ok=True)

# Save the cleaned Metadata df
subset_path = os.path.join(output_dir, "fakeddit_subset.csv")

image_df.to_csv(subset_path, index=False)
print(f"Saved clean metadata ({len(image_df)} rows) to: {subset_path}")

# Save the Concept List
concepts_path = os.path.join(output_dir, "concepts_list.json")
with open(concepts_path, 'w') as f:
    json.dump(concepts, f)
print(f"Saved concept list ({len(concepts)} labels) to: {concepts_path}")

# Save the Concept Embedding Matrix
concept_emb_path = os.path.join(output_dir, "concept_embeddings.npy")
np.save(concept_emb_path, concept_embs)
print(f"Saved concept embedding matrix (Shape: {concept_embs.shape}) to: {concept_emb_path}")


In [ ]:
# ZIP the image folder for sharing
!zip -r {output_dir}/fakeddit_images.zip {img_dir}

In [ ]:
# Zip the whole assets folder
zip_path = os.path.join("/content", "master_pipeline_assets.zip")
!zip -r {zip_path} {output_dir}

# Download the zipped folder
files.download(zip_path)